In [1]:
import rpy2.robjects.packages as rpackages
import rpy2.robjects as ro

In [2]:
rpackages.importr('DBI')
rpackages.importr('lme4')
rpackages.importr('DT')

rpy2.robjects.packages.Package as a <module 'DT'>

In [3]:
ro.r('''
           RESULTS_DB_PATH <- "/home/abishekthamma/PycharmProjects/masters_thesis/ss-llm/nanoGPT/results/results.db"
results_db <- dbConnect(RSQLite::SQLite(), RESULTS_DB_PATH)

baseline_df <- dbGetQuery(results_db,"SELECT SPRTNaturalStories.RTUID, 
SPRTNaturalStories.WorkerID, 
SPRTNaturalStories.StoryWordID, 
SPRTNaturalStories.RT, 
WordDetails.Word as WordCategory, 
WordDetails.CharacterLength, 
WordDetails.WordUID as WordCategoryID,
WordDetails.LogFrequencies as LogFrequencies,
Story.POSTag as POSTag
FROM SPRTNaturalStories 
JOIN Story on SPRTNaturalStories.StoryWordID = Story.StoryWordID 
JOIN WordDetails on WordDetails.WordUID = Story.WordUID 
")
'''
)
ro.r("baseline_df")

R[write to console]: In addition: 
R[write to console]: Warning messages:

R[write to console]: 1: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages

R[write to console]: 2: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages

R[write to console]: 3: 
R[write to console]: In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
R[write to console]: 
 
R[write to console]:  library ‘/usr/lib/R/site-library’ contains no packages



RTUID,WorkerID,StoryWordID,...,WordCategoryID,LogFrequencies,POSTag
1,'A3QJPB0N...,1,...,1,5.256744,'ADP'
2,'A2RPQGUW...,1,,1,5.256744,'ADP'
3,'A11KMPAZ...,1,,1,5.256744,'ADP'
4,'A1U1QL61...,1,,1,5.256744,'ADP'
...,...,...,,...,...,...
848764,'A253Q11T...,10256,,2141,0.000000,'NOUN'
848765,'A1WURYN1...,10256,,2141,0.000000,'NOUN'
848766,'A1INWCFG...,10256,,2141,0.000000,'NOUN'
848767,'A279TEEN...,10256,,2141,0.000000,'NOUN'


In [4]:
ro.r('''
baseline_df$LogRT <- log(baseline_df$RT)

baseline_df$WorkerID <- as.factor(baseline_df$WorkerID)
baseline_df$WordCategoryID <- as.factor(baseline_df$WordCategoryID)
baseline_df$POSTag <- as.factor(baseline_df$POSTag)
baseline_df$CharacterLength_c <- scale(baseline_df$CharacterLength)

#Should I scale Log Frequencies(?)
baseline_df$LogFrequencies_c <- scale(baseline_df$LogFrequencies)

''')

In [5]:
ro.r("baseline_df")

RTUID,WorkerID,StoryWordID,...,LogRT,CharacterLength_c,LogFrequencies_c
1,A3QJP...,1,...,6.828712,-1.062253,0.595970
2,A2RPQ...,1,,6.161207,-1.062253,0.595970
3,A11KM...,1,,5.605802,-1.062253,0.595970
4,A1U1Q...,1,,5.869297,-1.062253,0.595970
...,...,...,,...,...,...
848764,A253Q...,10256,,6.684612,2.472302,-2.888350
848765,A1WUR...,10256,,6.804615,2.472302,-2.888350
848766,A1INW...,10256,,6.356108,2.472302,-2.888350
848767,A279T...,10256,,7.020191,2.472302,-2.888350


In [6]:
ro.r('''
exp_index <- sample(1:nrow(baseline_df), nrow(baseline_df)*0.5)

baseline_df$exp <- 0
baseline_df$exp[exp_index] <- 1
baseline_df$exp <- as.factor(baseline_df$exp)

exploratory_df <- baseline_df[baseline_df$exp == 1,]
test_df <- baseline_df[baseline_df$exp == 0,]
''')

In [8]:
#Find correlation between LogFrequencies_c and CharacterLength_c
ro.r(''' summary(cor(baseline_df$LogFrequencies_c, baseline_df$CharacterLength_c))''')


'Min....,'1st ...,'Medi...,'Mean...,'3rd ...,'Max....


In [74]:

ro.r(f'''
fit_formula <- "LogRT ~ 1  + ( 1 | WorkerID) + (1 | WordCategoryID)"
fit <- lmer(fit_formula, data=exploratory_df, REML=F)
fit_summary <- summary(fit)
''')

In [104]:
from rpy2.robjects import pandas2ri
import pandas as pd
pandas2ri.activate()


fixed_effects, random_effects, fit_stats, var_cov_matrix = get_model_summary('fit', 'fit_summary')
fixed_effects



Model Fit Statistics:
              AIC           BIC  Log-Likelihood
0  163746.265924  163790.09949   -81869.132962

Variance-Covariance Matrix of Random Effects:
               grp         var1  var2      vcov     sdcor
1  WordCategoryID  (Intercept)  None  0.008336  0.091301
2        WorkerID  (Intercept)  None  0.061081  0.247147
3        Residual         None  None  0.084840  0.291274


,Estimate,Std. Error,t value
(Intercept),5.751873,0.01857,309.743399


In [105]:
random_effects

,grp,var1,var2,vcov,sdcor
1,WordCategoryID,(Intercept),None,0.008336,0.091301
2,WorkerID,(Intercept),None,0.061081,0.247147
3,Residual,None,None,0.084840,0.291274


In [106]:
fit_stats

,AIC,BIC,Log-Likelihood
0,163746.265924,163790.09949,-81869.132962


In [107]:
var_cov_matrix

,grp,var1,var2,vcov,sdcor
1,WordCategoryID,(Intercept),None,0.008336,0.091301
2,WorkerID,(Intercept),None,0.061081,0.247147
3,Residual,None,None,0.084840,0.291274


In [116]:

def extract_model_summary(model_name, summary_name):

    # 1. Fixed Effects
    fixed_effects = ro.r(f'as.data.frame({summary_name}$coefficients)')
    fixed_effects_df = pandas2ri.rpy2py(fixed_effects)
    #print(fixed_effects_df)
    fixed_effects_df.columns = ['Estimate', 'Std. Error', 't value'] #, 'Pr(>|t|)']

    # 2. Random Effects
    random_effects = ro.r(f'as.data.frame(VarCorr({model_name}))')
    random_effects_df = pandas2ri.rpy2py(random_effects)
    #print("\nRandom Effects (Variance and Std. Dev by Group):\n", random_effects_df)

    # 3. Residuals
    residuals = ro.r(f'as.data.frame({summary_name}$residuals)')
    residuals_df = pandas2ri.rpy2py(residuals)
    #print("\nResiduals:\n", residuals_df)

    # 4. Model Fit Statistics
    aic = ro.r(f'AIC({model_name})')[0]
    bic = ro.r(f'BIC({model_name})')[0]
    log_likelihood = ro.r(f'logLik({model_name})')[0]
    warnings = ro.r('warnings()') 
    fit_stats_df = pd.DataFrame({
        'AIC': [aic],
        'BIC': [bic],
        'Log-Likelihood': [log_likelihood],
        'Warnings': [warnings]
    })

    # 5. Variance-Covariance Matrix of Random Effects
    var_cov_matrix = ro.r(f'as.data.frame({summary_name}$varcor)')
    var_cov_matrix_df = pandas2ri.rpy2py(var_cov_matrix)
    
    return fixed_effects_df, random_effects_df, fit_stats_df, var_cov_matrix_df

def execute_r_lmer_model(fit_formula, data_frame_name, fit_name, fit_summary_name):
    ro.r(f'''
    {fit_name} <- lmer({fit_formula}, data={data_frame_name}, REML=F)
    {fit_summary_name} <- summary({fit_name})
    ''')
    
    fixed_effects, random_effects, fit_stats, var_cov_matrix = extract_model_summary(fit_name, fit_summary_name)

    return fixed_effects, random_effects, fit_stats, var_cov_matrix



In [117]:
fit_name = "fit"
fit_summary_name = "fit_summary"
data_frame_name = "exploratory_df"
fit_formula = "LogRT ~ CharacterLength_c  + ( 1 | WorkerID) + (1 | WordCategoryID)"


fixed_effects, random_effects, fit_stats, var_cov_matrix = execute_r_lmer_model(fit_formula,data_frame_name,  fit_name, fit_summary_name)

print(fixed_effects)
print(random_effects)
print(fit_stats)
print(var_cov_matrix)

                   Estimate  Std. Error     t value
(Intercept)        5.721652    0.018630  307.127103
CharacterLength_c  0.033020    0.001778   18.573886
              grp         var1  var2      vcov     sdcor
1  WordCategoryID  (Intercept)  None  0.007040  0.083907
2        WorkerID  (Intercept)  None  0.061113  0.247210
3        Residual         None  None  0.084843  0.291279
             AIC         BIC  Log-Likelihood
0  163426.866042  163481.658   -81708.433021
              grp         var1  var2      vcov     sdcor
1  WordCategoryID  (Intercept)  None  0.007040  0.083907
2        WorkerID  (Intercept)  None  0.061113  0.247210
3        Residual         None  None  0.084843  0.291279


In [122]:
fit_name = "fit2"
fit_summary_name = "fit_summary2"
data_frame_name = "exploratory_df"

fit_formula = "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 | WorkerID) + (1 | POSTag)"


fixed_effects, random_effects, fit_stats, var_cov_matrix = execute_r_lmer_model(fit_formula, data_frame_name, fit_name, fit_summary_name)

print(fixed_effects)
print(random_effects)
print(fit_stats)
print(var_cov_matrix)

                   Estimate  Std. Error     t value
(Intercept)        5.738427    0.025361  226.270378
CharacterLength_c  0.027061    0.000560   48.362314
        grp         var1  var2      vcov     sdcor
1  WorkerID  (Intercept)  None  0.060746  0.246467
2    POSTag  (Intercept)  None  0.003319  0.057612
3  Residual         None  None  0.087466  0.295747
             AIC            BIC  Log-Likelihood
0  171695.944476  171750.736434   -85842.972238
        grp         var1  var2      vcov     sdcor
1  WorkerID  (Intercept)  None  0.060746  0.246467
2    POSTag  (Intercept)  None  0.003319  0.057612
3  Residual         None  None  0.087466  0.295747


In [137]:
fit_formula_list = [
    "LogRT ~ CharacterLength_c  + ( 1 | WorkerID) + (1 | WordCategoryID)",
    "LogRT ~ CharacterLength_c  + ( 1 | WorkerID) + (1 | POSTag)",
    
    "LogRT ~ LogFrequencies_c + ( 1 | WorkerID) + (1 | WordCategoryID)",
    "LogRT ~ LogFrequencies_c + ( 1 | WorkerID) + (1 | POSTag)",
    
    "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 | WorkerID) + (1 | WordCategoryID)",
    "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 | WorkerID) + (1 | POSTag)",
    
    "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + CharacterLength_c | WorkerID) + (1 | WordCategoryID)",
    "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + CharacterLength_c | WorkerID) + (1 | POSTag)",

    "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + LogFrequencies_c | WorkerID) + (1 | WordCategoryID)",
    "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + LogFrequencies_c | WorkerID) + (1 | POSTag)",

    "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + CharacterLength_c + LogFrequencies_c | WorkerID) + (1 | WordCategoryID)",
    "LogRT ~ 1 + ( 1 + CharacterLength_c + LogFrequencies_c | WorkerID) + (1 | WordCategoryID)",    
    "LogRT ~ 1 + ( 1 + CharacterLength_c + LogFrequencies_c | WorkerID) + (1 | POSTag)",
    "LogRT ~ CharacterLength_c + LogFrequencies_c + ( 1 + CharacterLength_c + LogFrequencies_c | WorkerID) + (1 | POSTag)",


]

In [138]:
from tqdm import tqdm
import time
results_list = []

for fit_formula in tqdm(fit_formula_list):
    start_time = time.time()
    fixed_effects, random_effects, fit_stats, var_cov_matrix = execute_r_lmer_model(fit_formula, data_frame_name, fit_name, fit_summary_name)
    
    results_list.append({"fit_formula": fit_formula, 
                         "fixed_effects": fixed_effects, 
                         "random_effects": random_effects, 
                         "fit_stats": fit_stats,
                         "var_cov_matrix": var_cov_matrix,
                         "execution_time": time.time() - start_time})


results_list

  0%|          | 0/3 [00:00<?, ?it/s]R[write to console]: In addition: 
R[write to console]: Warning message:

R[write to console]: In checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv,  :
R[write to console]: 
 
R[write to console]:  Model failed to converge with max|grad| = 0.00397905 (tol = 0.002, component 1)

 33%|███▎      | 1/3 [05:35<11:10, 335.21s/it]R[write to console]: In addition: 
R[write to console]: Warning message:

R[write to console]: In checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv,  :
R[write to console]: 
 
R[write to console]:  Model failed to converge with max|grad| = 0.0128911 (tol = 0.002, component 1)

100%|██████████| 3/3 [10:28<00:00, 209.60s/it]


[{'fit_formula': 'LogRT ~ 1 + ( 1 + CharacterLength_c + LogFrequencies_c | WorkerID) + (1 | WordCategoryID)',
  'fixed_effects':              Estimate  Std. Error     t value
  (Intercept)   5.61817    0.019124  293.779003,
  'random_effects':               grp               var1               var2      vcov     sdcor
  1  WordCategoryID        (Intercept)               None  0.006672  0.081683
  2        WorkerID        (Intercept)               None  0.071306  0.267033
  3        WorkerID  CharacterLength_c               None  0.000372  0.019278
  4        WorkerID   LogFrequencies_c               None  0.000555  0.023568
  5        WorkerID        (Intercept)  CharacterLength_c  0.002026  0.393496
  6        WorkerID        (Intercept)   LogFrequencies_c -0.003187 -0.506457
  7        WorkerID  CharacterLength_c   LogFrequencies_c -0.000300 -0.659293
  8        Residual               None               None  0.084373  0.290470,
  'fit_stats':              AIC            BIC  Log-Lik

In [151]:
ro.r('warnings()')

ValueError: Not an R object.

<rpy2.robjects.vectors.ListVector object at 0x7eadc57b8740> [RTYPES.VECSXP]
R classes: ('warnings',)
[LangSexpVector]
  Model failed to converge with max|grad| = 0.0128911 (tol = 0.002, component 1): <class 'rpy2.rinterface.LangSexpVector'>
  <rpy2.rinterface.LangSexpVector object at 0x7eadb1da24c0> [RTYPES.LANGSXP]

In [141]:
len(results_list)

14

In [157]:
results_df = pd.DataFrame(results_list)
results_df["AIC"] = results_df["fit_stats"].apply(lambda x: x["AIC"].values[0])
results_df["BIC"] = results_df["fit_stats"].apply(lambda x: x["BIC"].values[0])
results_df["Log-Likelihood"] = results_df["fit_stats"].apply(lambda x: x["Log-Likelihood"].values[0])

In [158]:
results_df = results_df.drop(columns=["fit_stats"])
results_df

,fit_formula,fixed_effects,random_effects,var_cov_matrix,execution_time,AIC,BIC,Log-Likelihood
0,LogRT ~ 1 + ( 1 + CharacterLength_c + LogFrequ...,Estimate Std. Error t value ...,grp var1 ...,grp var1 ...,335.209563,161687.121799,161785.747324,-80834.560900
1,LogRT ~ 1 + ( 1 + CharacterLength_c + LogFrequ...,Estimate Std. Error t value (I...,grp var1 v...,grp var1 v...,115.139529,169040.543532,169139.169057,-84511.271766
2,LogRT ~ CharacterLength_c + LogFrequencies_c +...,Estimate Std. Error t ...,grp var1 v...,grp var1 v...,178.420686,168816.997152,168937.539460,-84397.498576
3,LogRT ~ CharacterLength_c + ( 1 | WorkerID) +...,Estimate Std. Error t ...,grp var1 var2 vcov...,grp var1 var2 vcov...,15.779711,163426.866042,163481.658000,-81708.433021
4,LogRT ~ CharacterLength_c + ( 1 | WorkerID) +...,Estimate Std. Error t ...,grp var1 var2 vcov s...,grp var1 var2 vcov s...,11.176801,171695.944476,171750.736434,-85842.972238
5,LogRT ~ LogFrequencies_c + ( 1 | WorkerID) + (...,Estimate Std. Error t v...,grp var1 var2 vcov...,grp var1 var2 vcov...,18.168842,163314.069746,163368.861704,-81652.034873
6,LogRT ~ LogFrequencies_c + ( 1 | WorkerID) + (...,Estimate Std. Error t v...,grp var1 var2 vcov s...,grp var1 var2 vcov s...,11.620378,170939.632793,170994.424751,-85464.816397
7,LogRT ~ CharacterLength_c + LogFrequencies_c +...,Estimate Std. Error t ...,grp var1 var2 vcov...,grp var1 var2 vcov...,20.015982,163253.665880,163319.416230,-81620.832940
8,LogRT ~ CharacterLength_c + LogFrequencies_c +...,Estimate Std. Error t ...,grp var1 var2 vcov s...,grp var1 var2 vcov s...,13.127882,170658.376033,170724.126383,-85323.188017
9,LogRT ~ CharacterLength_c + LogFrequencies_c +...,Estimate Std. Error t ...,grp var1 ...,grp var1 ...,86.378951,161711.526039,161799.193172,-80847.763020
